In [1]:
%%capture
!pip install --upgrade unsloth
!pip install transformers peft datasets trl -q

In [2]:
import os, sys, json, time, re, warnings, logging, torch
from pathlib import Path

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
print(f'PyTorch: {torch.__version__}')

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB
PyTorch: 2.10.0+cu128


In [ ]:
# Mount Google Drive — MUST run before any code creates paths under
# /content/drive, otherwise drive.mount() fails ('Mountpoint must not
# already contain files') or a stray local dir tricks a naive exists()
# check into skipping the real mount, silently writing checkpoints to
# ephemeral Colab storage instead of Drive.
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

In [3]:
# ====== SHARED CONFIG — chỉnh ở đây ======
CONFIG = {
    # Kaggle API
    'kaggle_username': 'thnhcngl',

    # Kaggle datasets
    'ds_sft_ckpt': 'thnhcngl/dvsktt-sft-best-checkpoint',
    'ds_sft_data': 'thnhcngl/dvsktt-ner-sft',
    'ds_han_data': 'thnhcngl/dvsktt-han-pretrain',

    # Local paths
    'data_dir':    '/content/data',
    'work_dir':    '/content/drive/MyDrive/dvsktt_ner',
    'ckpt_dir':    '/content/drive/MyDrive/dvsktt_ner/checkpoints',
    'result_dir':  '/content/drive/MyDrive/dvsktt_ner/results',
    'log_dir':     '/content/drive/MyDrive/dvsktt_ner/logs',

    # Model
    'base_model':   'unsloth/qwen2.5-7b-unsloth-bnb-4bit',
    'max_seq_len':  512,
    'lora_rank':    16,
    'lora_alpha':   32,
    'lora_dropout': 0.05,

    # Pretrain
    'pretrain_lr':     5e-5,
    'pretrain_epochs': 1,
    'pretrain_batch':  2,
    'pretrain_sample': 5000,
    'pretrain_grad_accum': 4,

    # SFT
    'sft_lr':          5e-5,
    'sft_epochs':      3,
    'sft_batch':       1,
    'sft_grad_accum':  4,

    # Evaluate
    'eval_batch':      4,
    'max_new_tokens':  600,
    'save_every':      50,

    # Resume
    'resume_from_step': 0,  # đặt số step để resume, 0 = train từ đầu
}

# Tạo thư mục
for d in ['data_dir','work_dir','ckpt_dir','result_dir','log_dir']:
    os.makedirs(CONFIG[d], exist_ok=True)

print('Config loaded. Dirs created.')
print(f"Work dir: {CONFIG['work_dir']}")

Config loaded. Dirs created.
Work dir: /content/drive/MyDrive/dvsktt_ner


In [ ]:
# Setup Kaggle API — credentials are NEVER hardcoded here.
# Set them as Colab Secrets (key icon in the left sidebar) named
# KAGGLE_USERNAME / KAGGLE_KEY, or export them as env vars if running elsewhere.
import os

try:
    from google.colab import userdata
    os.environ.setdefault('KAGGLE_USERNAME', userdata.get('KAGGLE_USERNAME'))
    os.environ.setdefault('KAGGLE_KEY', userdata.get('KAGGLE_KEY'))
except Exception:
    pass

if not os.environ.get('KAGGLE_USERNAME') or not os.environ.get('KAGGLE_KEY'):
    raise RuntimeError(
        'Missing Kaggle credentials. Set KAGGLE_USERNAME/KAGGLE_KEY as Colab Secrets '
        '(key icon in sidebar) or environment variables before running this cell.'
    )

print('Kaggle API configured via environment variables.')
!kaggle --version

In [ ]:
import subprocess

def download_dataset(ds_name, data_dir, max_retries=4, retry_delay=15):
    name = ds_name.split('/')[-1]
    dest = f'{data_dir}/{name}'

    if os.path.exists(dest) and len(os.listdir(dest)) > 0:
        print(f'Already exists: {dest}')
        return dest

    os.makedirs(dest, exist_ok=True)

    # Kaggle API rate-limits rapid successive calls (403 Forbidden on
    # GetDatasetMetadata) — retry with backoff instead of failing fast.
    r = None
    for attempt in range(1, max_retries + 1):
        print(f'Downloading {ds_name} (attempt {attempt}/{max_retries})...')
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', ds_name, '-p', dest, '--unzip'],
            capture_output=True, text=True
        )
        print(r.stdout[:200])
        if r.returncode == 0:
            break
        print(f'ERROR: {r.stderr[:200]}')
        if attempt < max_retries:
            print(f'Retrying in {retry_delay}s...')
            time.sleep(retry_delay)
    if r is None or r.returncode != 0:
        return None

    print(f'Done: {dest}')
    for f in os.listdir(dest):
        print(f'  {f}')
    return dest

# dvsktt-han-pretrain và dvsktt-ner-sft là dataset Private trên Kaggle và
# liên tục bị 403 Forbidden qua API (không phải rate-limit — đã retry vẫn fail)
# nhưng cả 2 đã có sẵn ngay trong repo này (data/raw/), nên đọc thẳng từ đó,
# khỏi cần gọi Kaggle API cho 2 dataset này. Chỉ checkpoint (quá lớn cho git)
# mới cần tải qua Kaggle.
sft_ckpt_path = download_dataset(CONFIG['ds_sft_ckpt'], CONFIG['data_dir'])
REPO_DATA_DIR = '/content/repo/data/raw'
han_data_path = f'{REPO_DATA_DIR}/han_pretrain'
sft_data_path = f'{REPO_DATA_DIR}/ner_sft'


In [10]:
# ====== Logger — ghi log ra file + console ======
class Logger:
    def __init__(self, log_path):
        self.log_path = log_path
        self.start    = time.time()
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, 'a') as f:
            f.write(f'\n===== Session started: {time.strftime("%Y-%m-%d %H:%M:%S")} =====\n')

    def log(self, msg, also_print=True):
        elapsed = time.time() - self.start
        line    = f'[{elapsed:>8.1f}s] {msg}'
        with open(self.log_path, 'a') as f:
            f.write(line + '\n')
        if also_print:
            print(line)

    def save_state(self, state, name):
        path = os.path.join(os.path.dirname(self.log_path), f'{name}.json')
        with open(path, 'w') as f:
            json.dump(state, f, ensure_ascii=False, indent=2)
        self.log(f'State saved: {path}')
        return path

    def load_state(self, name):
        path = os.path.join(os.path.dirname(self.log_path), f'{name}.json')
        if os.path.exists(path):
            with open(path) as f:
                state = json.load(f)
            self.log(f'State loaded: {path}')
            return state
        return None

print('Logger ready.')

Logger ready.


## Load Model từ Pretrain Checkpoint

In [11]:
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
import torch.optim as optim
from unsloth import FastLanguageModel

logger = Logger(f"{CONFIG['log_dir']}/sft.log")

resume_state = logger.load_state('sft_state')
RESUME_EPOCH = resume_state['epoch'] if resume_state else 0
RESUME_STEP  = resume_state['step']  if resume_state else 0

if RESUME_STEP > 0:
    logger.log(f'Resuming SFT from epoch {RESUME_EPOCH+1}, step {RESUME_STEP}')
else:
    logger.log('Starting fresh SFT')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[     0.0s] Starting fresh SFT


In [12]:
# Load từ pretrain checkpoint hoặc SFT checkpoint khi resume
if RESUME_STEP > 0:
    model_path = resume_state.get('ckpt', f"{CONFIG['ckpt_dir']}/pretrain_final")
else:
    model_path = f"{CONFIG['ckpt_dir']}/pretrain_final"
    if not os.path.exists(model_path):
        # Fallback: load từ Kaggle checkpoint
        model_path = sft_ckpt_path
        logger.log(f'Pretrain ckpt not found, using: {model_path}')

logger.log(f'Loading model: {model_path}')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_path,
    max_seq_length = CONFIG['max_seq_len'],
    load_in_4bit   = True,
    dtype          = None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r              = CONFIG['lora_rank'],
    lora_alpha     = CONFIG['lora_alpha'],
    lora_dropout   = CONFIG['lora_dropout'],
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
logger.log('Model loaded!')
model.print_trainable_parameters()

[     0.0s] Pretrain ckpt not found, using: /content/data/dvsktt-sft-best-checkpoint
[     0.0s] Loading model: /content/data/dvsktt-sft-best-checkpoint
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/106k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
Unsloth: Already have LoRA adapters! We shall skip this step.


[    55.9s] Model loaded!
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## Load & Tokenize SFT Data

In [13]:
def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(l) for l in f]

train_data = load_jsonl(f'{sft_data_path}/train.jsonl')
logger.log(f'Train records: {len(train_data):,}')

def format_prompt(record):
    return (
        f"### Instruction:\n{record['instruction']}\n\n"
        f"### Input:\n{record['input']}\n\n"
        f"### Output:\n{record['output']}"
        f"{tokenizer.eos_token}"
    )

train_texts   = [format_prompt(r) for r in train_data]
train_dataset = Dataset.from_dict({'text': train_texts})

def tokenize_sft(examples):
    result = tokenizer(
        examples['text'],
        truncation = True,
        max_length = CONFIG['max_seq_len'],
        padding    = 'max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

train_tokenized = train_dataset.map(tokenize_sft, batched=True, remove_columns=['text'], num_proc=2)
sft_collator    = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)
logger.log(f'SFT tokenized: {len(train_tokenized):,} examples')

[    56.0s] Train records: 5,600


Map (num_proc=2):   0%|          | 0/5600 [00:00<?, ? examples/s]

[    61.8s] SFT tokenized: 5,600 examples


## SFT Training Loop

In [14]:
EPOCHS     = CONFIG['sft_epochs']
BATCH      = CONFIG['sft_batch']
GRAD_ACCUM = CONFIG['sft_grad_accum']
SAVE_EVERY = 200

sft_loader  = DataLoader(train_tokenized, batch_size=BATCH, shuffle=True, collate_fn=sft_collator)
total_steps = len(sft_loader) * EPOCHS
optimizer   = optim.AdamW(model.parameters(), lr=CONFIG['sft_lr'])
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = int(total_steps * 0.05),
    num_training_steps = total_steps,
)

logger.log(f'SFT total steps: {total_steps:,} | Resume from step: {RESUME_STEP}')

model.train()
global_step = 0
best_loss   = resume_state.get('best_loss', float('inf')) if resume_state else float('inf')

for epoch in range(RESUME_EPOCH, EPOCHS):
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(sft_loader):
        global_step += 1

        if global_step <= RESUME_STEP:
            continue

        batch   = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)

        # Detect NaN
        if torch.isnan(outputs.loss):
            logger.log(f'NaN loss at step {global_step}! Stopping epoch.')
            break

        loss = outputs.loss / GRAD_ACCUM
        loss.backward()
        total_loss += outputs.loss.item()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        if global_step % 100 == 0:
            avg     = total_loss / (step + 1)
            elapsed = time.time() - logger.start
            eta     = elapsed / (global_step - RESUME_STEP) * (total_steps - global_step)
            logger.log(
                f'Epoch {epoch+1}/{EPOCHS} | Step {global_step}/{total_steps} | '
                f'Loss: {avg:.4f} | ETA: {eta/3600:.2f}h'
            )

        if global_step % SAVE_EVERY == 0:
            avg_loss  = total_loss / (step + 1)
            ckpt_path = f"{CONFIG['ckpt_dir']}/sft_step{global_step}"
            model.save_pretrained(ckpt_path)
            tokenizer.save_pretrained(ckpt_path)
            logger.save_state({
                'epoch':     epoch,
                'step':      global_step,
                'loss':      avg_loss,
                'best_loss': best_loss,
                'ckpt':      ckpt_path,
            }, 'sft_state')
            logger.log(f'Checkpoint saved: {ckpt_path}')

    avg_epoch = total_loss / len(sft_loader)
    logger.log(f'Epoch {epoch+1} done | Avg Loss: {avg_epoch:.4f}')

    if avg_epoch < best_loss:
        best_loss  = avg_epoch
        best_path  = f"{CONFIG['ckpt_dir']}/sft_best"
        model.save_pretrained(best_path)
        tokenizer.save_pretrained(best_path)
        logger.log(f'Best model saved: {best_path} (loss: {best_loss:.4f})')

final_path = f"{CONFIG['ckpt_dir']}/sft_final"
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
logger.save_state({'step': total_steps, 'status': 'completed', 'best_loss': best_loss}, 'sft_state')
logger.log(f'SFT complete! Best loss: {best_loss:.4f}')

`use_return_dict` is deprecated! Use `return_dict` instead!


[    61.9s] SFT total steps: 16,800 | Resume from step: 0
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
[   119.2s] Epoch 1/3 | Step 100/16800 | Loss: 1.3306 | ETA: 5.53h
[   167.7s] Epoch 1/3 | Step 200/16800 | Loss: 1.3177 | ETA: 3.87h


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dvsktt_ner/checkpoints/sft_step200/tokenizer_config.json.


[   169.6s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/sft_state.json
[   169.6s] Checkpoint saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/sft_step200
[   217.8s] Epoch 1/3 | Step 300/16800 | Loss: 1.2943 | ETA: 3.33h
[   266.1s] Epoch 1/3 | Step 400/16800 | Loss: 1.2911 | ETA: 3.03h


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dvsktt_ner/checkpoints/sft_step400/tokenizer_config.json.


...

[  8192.4s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/sft_state.json
[  8192.4s] Checkpoint saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/sft_step16600
[  8240.7s] Epoch 3/3 | Step 16700/16800 | Loss: 0.2160 | ETA: 0.01h
[  8289.2s] Epoch 3/3 | Step 16800/16800 | Loss: 0.2167 | ETA: 0.00h


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dvsktt_ner/checkpoints/sft_step16800/tokenizer_config.json.


[  8290.5s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/sft_state.json
[  8290.5s] Checkpoint saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/sft_step16800
[  8290.5s] Epoch 3 done | Avg Loss: 0.2167
[  8291.9s] Best model saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/sft_best (loss: 0.2167)


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dvsktt_ner/checkpoints/sft_final/tokenizer_config.json.


[  8293.1s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/sft_state.json
[  8293.1s] SFT complete! Best loss: 0.2167
